In [ ]:
# dependencies
%pip -q install --upgrade pip
%pip -q install --upgrade --force-reinstall --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip -q install "transformers>=4.57.0,<4.58.0" "trl>=0.24.0,<0.25.0" "peft>=0.18.0,<0.19.0" "accelerate>=1.4.0,<2.0.0" "bitsandbytes==0.49.2" "datasets>=4.0.0,<5.0.0" "huggingface-hub>=0.36.0,<1.0.0"

In [ ]:
# simple paths and optional Hugging Face login
from pathlib import Path
import json
import os

from huggingface_hub import login

DATA_DIR = Path("/content/wos_orchestration")
OUTPUT_DIR = DATA_DIR / "outputs"
REPORTS_DIR = DATA_DIR / "reports"
for path in (DATA_DIR, OUTPUT_DIR, REPORTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATA_DIR / "qwen3_orchestrator_dataset.cleaned.jsonl"
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Expected cleaned dataset at {DATASET_PATH}. Put the cleaned JSONL in /content/wos_orchestration before training."
    )

HF_MODEL_REPO = "YOUR_HF_USERNAME/wos_orch_qwen"
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=True, skip_if_logged_in=True)

print(json.dumps({
    "dataset_path": str(DATASET_PATH),
    "output_dir": str(OUTPUT_DIR),
    "hf_model_repo": HF_MODEL_REPO,
    "hf_token_loaded": bool(HF_TOKEN),
}, indent=2))

In [ ]:
# load the Qwen3 32B model and attach LoRA adapters
import torch
from unsloth import FastLanguageModel

# Qwen3-32B is already the post-trained instruction-following checkpoint.
# Unsloth does not publish a separate 32B '-Instruct' repo.
MODEL_NAME = "unsloth/Qwen3-32B-bnb-4bit"
ABSOLUTE_MAX_SEQ_LENGTH = 8192
LOAD_IN_4BIT = True
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.0

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=ABSOLUTE_MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    full_finetuning=False,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "<|PAD_TOKEN|>"
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(json.dumps({
    "model_name": MODEL_NAME,
    "load_in_4bit": LOAD_IN_4BIT,
    "absolute_max_seq_length": ABSOLUTE_MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "note": "Qwen3-32B already includes post-training/instruction behavior; the chat template remains Qwen3-compatible.",
}, indent=2))

In [ ]:
# load the cleaned dataset, run a small sanity check, and render chat text for training
import hashlib
import json
import math
from datasets import Dataset

PLANNING_PREFIXES = (
    "i need to",
    "i'll",
    "i will",
    "let me",
    "first, i",
    "first i",
    "now i need to",
    "the user wants",
    "we need to",
)
PLANNING_SUBSTRINGS = (
    "i need to ask the user",
    "i need to clarify",
    "need clarification",
    "must ask the user",
    "must clarify",
    "before i can",
    "to proceed, i need",
    "i'll ask the user",
    "i will ask the user",
)

def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def stable_bucket(row: dict) -> float:
    row_id = str(row.get("id") or json.dumps(row, sort_keys=True))
    digest = hashlib.sha1(row_id.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) / 0xFFFFFFFF

def split_name(row: dict) -> str:
    metadata = row.get("metadata") or {}
    return str(metadata.get("split") or "").strip().lower()

def looks_like_unresolved_planning(text: str) -> bool:
    normalized = " ".join(text.strip().lower().split())
    if not normalized:
        return False
    return normalized.startswith(PLANNING_PREFIXES) or any(
        token in normalized for token in PLANNING_SUBSTRINGS
    )

def semantic_stats(rows: list[dict]) -> dict:
    unresolved_planning = 0
    reasoning_equals_content = 0
    for row in rows:
        final_msg = row["messages"][-1]
        content = (final_msg.get("content") or "").strip()
        reasoning = (final_msg.get("reasoning_content") or "").strip()
        if reasoning and content and reasoning == content:
            reasoning_equals_content += 1
        if looks_like_unresolved_planning(content):
            unresolved_planning += 1
    return {
        "unresolved_planning": unresolved_planning,
        "final_reasoning_equals_content": reasoning_equals_content,
    }

def render_record(row: dict) -> dict:
    text = tokenizer.apply_chat_template(
        row["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    token_count = len(
        tokenizer.apply_chat_template(
            row["messages"],
            tokenize=True,
            add_generation_prompt=False,
        )
    )
    return {"text": text, "token_count": token_count}

records = load_jsonl(DATASET_PATH)
sanity = semantic_stats(records)
if sanity["unresolved_planning"] or sanity["final_reasoning_equals_content"]:
    raise RuntimeError(f"Dataset failed sanity checks: {json.dumps(sanity)}")

train_records = [row for row in records if split_name(row) == "train"]
val_records = [row for row in records if split_name(row) in {"validation", "val", "eval", "test"}]

if not train_records:
    raise RuntimeError("No train rows found in dataset metadata.")
if not val_records:
    val_records = [row for row in train_records if stable_bucket(row) < 0.02]
    train_records = [row for row in train_records if stable_bucket(row) >= 0.02]

train_rows = [render_record(row) for row in train_records]
val_rows = [render_record(row) for row in val_records]

all_lengths = [row["token_count"] for row in train_rows + val_rows]
observed_max = max(all_lengths)
MAX_SEQ_LENGTH = min(ABSOLUTE_MAX_SEQ_LENGTH, int(math.ceil(observed_max / 256.0) * 256))

train_rows = [row for row in train_rows if row["token_count"] <= MAX_SEQ_LENGTH]
val_rows = [row for row in val_rows if row["token_count"] <= MAX_SEQ_LENGTH]

train_dataset = Dataset.from_list([{"text": row["text"]} for row in train_rows])
val_dataset = Dataset.from_list([{"text": row["text"]} for row in val_rows])

length_stats = {
    "rows": len(records),
    "train_examples": len(train_dataset),
    "validation_examples": len(val_dataset),
    "max_seq_length": MAX_SEQ_LENGTH,
    "observed_max_tokens": observed_max,
    "sanity": sanity,
}
(REPORTS_DIR / "length_stats.json").write_text(json.dumps(length_stats, indent=2), encoding="utf-8")
print(json.dumps(length_stats, indent=2))

In [ ]:
# configure training and start when ready
from trl import SFTConfig, SFTTrainer

START_TRAINING_NOW = False
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 1e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        output_dir=str(OUTPUT_DIR),
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        packing=False,
        eos_token="<|im_end|>",
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.03,
        weight_decay=0.01,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        optim="paged_adamw_8bit",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        save_total_limit=2,
        report_to="none",
    ),
)

if START_TRAINING_NOW:
    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(str(OUTPUT_DIR))
else:
    print("Training is configured. Set START_TRAINING_NOW = True to begin.")

In [ ]:
# optional merged export for faster inference
SAVE_MERGED_16BIT = False
MERGED_DIR = DATA_DIR / "merged-16bit"

if SAVE_MERGED_16BIT:
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")
    print(f"Saved merged model to {MERGED_DIR}")
else:
    print("Merged export is disabled. Set SAVE_MERGED_16BIT = True after training if you want a faster inference artifact.")

In [ ]:
# optional Hub push
PUSH_TO_HUB_NOW = False

if PUSH_TO_HUB_NOW:
    if not HF_TOKEN:
        raise ValueError("Set HF_TOKEN in the Colab environment before pushing.")
    model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
    print(f"Pushed model artifacts to {HF_MODEL_REPO}")
else:
    print("Hub push is disabled. Set PUSH_TO_HUB_NOW = True after training if you want to upload the result.")